In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
import seaborn as sns
import os

In [ ]:
# Load in ABCD and FC data, and set output file path
ABCD_DATA_DIR = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Data/abcd-data-release-5.1/core'
OUTPUT_DIR = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final'
FC_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/pMTG_FC_profiles_midb61.csv'

In [ ]:
# read in and clean/process ABCD data
def save_data(df, output_path, **kwargs):
    """
    Save data from a pandas DataFrame to a CSV file.
    """
    df.to_csv(output_path, **kwargs)

def load_data(data_path, columns=None, test=False, **kwargs):
    data = pd.read_csv(data_path, usecols=columns, **kwargs)
    return data

def preprocess_ravlt_data(ravlt_data):
    """
    Calculate RAVLT immediate and prepare the data for merging.
    If any NaN values are present in the trial data, no scores are calculated for that subject.

    Input: ravlt_data (DataFrame) - RAVLT data (including src_subject_id and all five trial columns)
    Output: ravlt_data_clean (DataFrame) - Cleaned RAVLT data with summary measures
    """

    ind_pea_ravlt = [
        'pea_ravlt_sd_trial_i_tc', 'pea_ravlt_sd_trial_ii_tc',
        'pea_ravlt_sd_trial_iii_tc', 'pea_ravlt_sd_trial_iv_tc',
        'pea_ravlt_sd_trial_v_tc'
    ]
    
    # Convert trial columns to numeric, coerce invalid values to NaN
    ravlt_data[ind_pea_ravlt] = ravlt_data[ind_pea_ravlt].apply(pd.to_numeric, errors='coerce')
    
    # Remove any rows with NaN values in the trial columns
    ravlt_data_clean = ravlt_data.dropna(subset=ind_pea_ravlt).copy()
    
    # Calculate RAVLT scores only for rows with complete data
    ravlt_data_clean['ravlt_immediate'] = ravlt_data_clean[ind_pea_ravlt].sum(axis=1)

    # Rename pea_ravlt_ld_trial_vi_tc and pea_ravlt_ld_trial_vii_tc to ravlt_short_delay and ravlt_long_delay
    ravlt_data_clean.rename(columns={
        'pea_ravlt_sd_trial_vi_tc': 'ravlt_short_delay',
        'pea_ravlt_ld_trial_vii_tc': 'ravlt_long_delay',
        'pea_ravlt_sd_listb_tc': 'ravlt_listb'
    }, inplace=True)

    # Remove all NaN or infinite values
    ravlt_data_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

    return ravlt_data_clean[['src_subject_id', 'ravlt_immediate', 'ravlt_short_delay', 'ravlt_long_delay']]

def merge_abcd_data(conn_path, abcd_path, output_path):
    """
    This function merges the ABCD demographic, NIH Toolbox, RAVLT, LMT, genetic, and connectivity data.
    
    To do:
    - Make more modular (e.g., function for loading, filtering, and cleaning data that can be repeated for each ABCD CSV file)
    - Add INR calculation function (rather than loading from old, hard-coded CSV: not done for data pre-September 14th, 2024)
    - Add arguments to:
        1. Specify the event names of interest
        2. Specify desired columns
        3. Specify the suffix for different timepoints (if longitudinal analyses will be done)
    """

    def rename_columns_with_suffix(df, suffix, ignore_cols=['src_subject_id']):
        """
        Add a suffix to column names, except for columns in ignore_cols.
        """
        return df.rename(columns=lambda x: x + suffix if x not in ignore_cols else x)

    ABCD_PATH = abcd_path

    # Load ABCD data files, with filtering for specific event names
    abcd_img = load_data(os.path.join(ABCD_PATH, 'imaging/mri_y_adm_info.csv'),
                         columns=['src_subject_id', 'eventname', 'mri_info_softwareversion', 'mri_info_studydate'])
    abcd_img = abcd_img[abcd_img['eventname'] == '2_year_follow_up_y_arm_1']
    abcd_img.drop(columns=['eventname'], inplace=True)  # Drop eventname after filtering

    fam_data = load_data(os.path.join(ABCD_PATH, 'abcd-general/abcd_y_lt.csv'),
                         columns=['src_subject_id', 'eventname', 'rel_family_id'])
    fam_data = fam_data[fam_data['eventname'] == 'baseline_year_1_arm_1']
    fam_data.drop(columns=['eventname'], inplace=True)  # Drop eventname after filtering

    site_age_data = load_data(os.path.join(ABCD_PATH, 'abcd-general/abcd_y_lt.csv'),
                              columns=['src_subject_id', 'eventname', 'site_id_l', 'interview_age'])
    site_age_data = site_age_data[site_age_data['eventname'] == '2_year_follow_up_y_arm_1']
    site_age_data.drop(columns=['eventname'], inplace=True)  # Drop eventname after filtering

    hand_data = load_data(os.path.join(ABCD_PATH, 'neurocognition/nc_y_ehis.csv'),
                          columns=['src_subject_id', 'eventname', 'ehi1b', 'ehi2b','ehi3b','ehi4b','ehi_y_ss_scoreb'])
    hand_data = hand_data[hand_data['eventname'] == 'baseline_year_1_arm_1']
    hand_data.drop(columns=['eventname'], inplace=True)  # Drop eventname after filtering

    sex_data = load_data(os.path.join(ABCD_PATH, 'gender-identity-sexual-health/gish_p_gi.csv'),
                         columns=['src_subject_id', 'eventname', 'demo_sex_v2'])
    sex_data = sex_data[sex_data['eventname'] == 'baseline_year_1_arm_1']
    sex_data.drop(columns=['eventname'], inplace=True)  # Drop eventname after filtering

    # Load the NIH Toolbox data, then add the '_2y' suffix after loading
    nihtb_data_2y = load_data(os.path.join(ABCD_PATH, 'neurocognition/nc_y_nihtb.csv'),
                columns=['src_subject_id', 'eventname',
                        'nihtbx_picvocab_v', 'nihtbx_picvocab_fc', 'nihtbx_picvocab_agecorrected', 'nihtbx_picvocab_uncorrected',
                        'nihtbx_reading_v', 'nihtbx_reading_fc', 'nihtbx_reading_agecorrected', 'nihtbx_reading_uncorrected',
                        'nihtbx_flanker_v', 'nihtbx_flanker_fc', 'nihtbx_flanker_agecorrected', 'nihtbx_flanker_uncorrected',
                        'nihtbx_pattern_v', 'nihtbx_pattern_fc', 'nihtbx_pattern_agecorrected', 'nihtbx_pattern_uncorrected',
                        'nihtbx_picture_v', 'nihtbx_picture_fc', 'nihtbx_picture_agecorrected', 'nihtbx_picture_uncorrected'])

    nihtb_data_2y = nihtb_data_2y[nihtb_data_2y['eventname'] == '2_year_follow_up_y_arm_1']
    nihtb_data_2y.drop(columns=['eventname'], inplace=True)  # Drop eventname after filtering
    nihtb_data_2y = rename_columns_with_suffix(nihtb_data_2y, '_2y')   

    # Get RAVLT summary data
    ravlt_data = load_data(os.path.join(ABCD_PATH, 'neurocognition/nc_y_ravlt.csv'),
                           columns=['src_subject_id', 'eventname',
                                    'pea_ravlt_sd_trial_i_tc', 'pea_ravlt_sd_trial_ii_tc', 'pea_ravlt_sd_trial_iii_tc', 
                                    'pea_ravlt_sd_trial_iv_tc', 'pea_ravlt_sd_trial_v_tc', 'pea_ravlt_sd_listb_tc', 'pea_ravlt_sd_trial_vi_tc','pea_ravlt_ld_trial_vii_tc','pea_ravlt_sd_listb_tc'])
    ravlt_data = ravlt_data[ravlt_data['eventname'] == '2_year_follow_up_y_arm_1']
    ravlt_data.drop(columns=['eventname'], inplace=True) 
    ravlt_summary = preprocess_ravlt_data(ravlt_data) 
    ravlt_summary = rename_columns_with_suffix(ravlt_summary, '_2y')

    # load 2 year SST data
    sst_data = load_data(os.path.join(ABCD_PATH, 'imaging/mri_y_tfmr_sst_beh.csv'),
                         columns=['src_subject_id', 'eventname', 'tfmri_sst_all_beh_total_issrt'])
    sst_data = sst_data[sst_data['eventname'] == '2_year_follow_up_y_arm_1']
    sst_data.drop(columns=['eventname'], inplace=True)

    # Load SES data 
    ses_data = load_data(os.path.join(ABCD_PATH, 'abcd-general/abcd_p_demo.csv'),
                         columns=['src_subject_id', 'eventname', 'demo_comb_income_v2', 'demo_roster_v2', 'demo_prnt_ed_v2_2yr_l', 'demo_prtnr_ed_v2_2yr_l',
                                  'demo_prnt_marital_v2', 'demo_prnt_prtnr_bio'])
    ses_data = ses_data[ses_data['eventname'] == 'baseline_year_1_arm_1'] # only available at baseline
    ses_data.drop(columns=['eventname'], inplace=True)

    # Load second language data 
    bilingual_data = load_data(os.path.join(ABCD_PATH, 'culture-environment/ce_y_acc.csv'),
                                columns=['src_subject_id', 'eventname', 'accult_q2_y'])
    bilingual_data = bilingual_data[bilingual_data['eventname'] == 'baseline_year_1_arm_1'] 
    bilingual_data.drop(columns=['eventname'], inplace=True)

    # Load dual language data
    dual_lang = load_data(os.path.join(ABCD_PATH, 'abcd-general/abcd_p_demo.csv'),
                          columns=['src_subject_id', 'eventname', 'demo_dual_lang_v2_l'])
    dual_lang = dual_lang[dual_lang['eventname'] == '1_year_follow_up_y_arm_1']
    dual_lang.drop(columns=['eventname'], inplace=True)

    motion_df = pd.read_csv(os.path.join(ABCD_DATA_DIR, 'imaging', 'mri_y_qc_motion.csv'))
    motion_df = motion_df[motion_df['eventname'] == '2_year_follow_up_y_arm_1']
    motion_df = motion_df[['src_subject_id', 'rsfmri_meanmotion','rsfmri_maxmotion','rsfmri_ntpoints', 'rsfmri_nvols', 'rsfmri_numtrs']]
    
    # Get Tanner puberty data 
    pds_full = pd.read_csv(os.path.join(ABCD_DATA_DIR, 'physical-health', 'ph_y_pds.csv'))
    baseline_sex = pds_full[pds_full['eventname'] == 'baseline_year_1_arm_1'][['src_subject_id', 'pds_sex_y']]
    tanner_df = pds_full[pds_full['eventname'] == '2_year_follow_up_y_arm_1'][
        ['src_subject_id', 'pds_bdyhair_y', 'pds_f4_2_y', 'pds_f5_y', 'pds_m4_y', 'pds_m5_y', 'pds_y_ss_female_category_2', 'pds_y_ss_male_cat_2']
    ]
    tanner_df = tanner_df.merge(baseline_sex, on='src_subject_id', how='left')

    def get_tanner_stage(row):
        if pd.notna(row['pds_y_ss_female_category_2']) and row['pds_sex_y'] == 2:
            return row['pds_y_ss_female_category_2']
        elif pd.notna(row['pds_y_ss_male_cat_2']) and row['pds_sex_y'] == 1:
            return row['pds_y_ss_male_cat_2']
        else:
            return None

    tanner_df['tanner_stage'] = tanner_df.apply(get_tanner_stage, axis=1)


    qa_df = pd.read_csv(os.path.join(ABCD_DATA_DIR, 'imaging', 'mri_y_qc_incl.csv'))
    qa_df = qa_df[qa_df['eventname'] == '2_year_follow_up_y_arm_1']
    qa_df = qa_df[['src_subject_id', 'imgincl_rsfmri_include']]

    # get data on race and ethnicity
    demo_df = pd.read_csv(os.path.join(ABCD_DATA_DIR, 'abcd-general', 'abcd_p_demo.csv'))
    demo_df = demo_df[demo_df['eventname'] == 'baseline_year_1_arm_1']
    demo_df = demo_df[['src_subject_id', 'demo_race_a_p___10', 'demo_race_a_p___11', 'demo_race_a_p___12', 'demo_race_a_p___13', 'demo_race_a_p___14', 'demo_race_a_p___15', 'demo_race_a_p___16', 'demo_race_a_p___17', 'demo_race_a_p___18', 'demo_race_a_p___19', 'demo_race_a_p___20', 'demo_race_a_p___21', 'demo_race_a_p___22', 'demo_race_a_p___23', 'demo_race_a_p___24', 'demo_race_a_p___25', 'demo_race_a_p___77', 'demo_race_a_p___99', 'demo_ethn_v2', 'demo_ethn2_v2']]

    # Merge all data
    combined_df = abcd_img.merge(fam_data, on='src_subject_id') \
                            .merge(site_age_data, on='src_subject_id') \
                            .merge(hand_data, on='src_subject_id') \
                            .merge(sex_data, on='src_subject_id') \
                            .merge(nihtb_data_2y, on='src_subject_id', how='left') \
                            .merge(ravlt_summary, on='src_subject_id', how='left') \
                            .merge(ses_data, on='src_subject_id', how='left') \
                            .merge(bilingual_data, on='src_subject_id', how='left') \
                            .merge(motion_df, on='src_subject_id', how='left') \
                            .merge(tanner_df, on='src_subject_id', how='left') \
                            .merge(qa_df, on='src_subject_id', how='left') \
                            .merge(sst_data, on='src_subject_id', how='left') \
                            .merge(demo_df, on='src_subject_id', how='left') \
                            .merge(dual_lang, on='src_subject_id', how='left') \
                            .drop_duplicates()

    # Print the columns to be combined with the connectivity data, without truncation
    pd.set_option('display.max_columns', None)
    print('Columns to be combined with connectivity data:', combined_df.columns)
    combined_df['src_subject_id'] = combined_df['src_subject_id'].str.replace('_', '', regex=False)
    
    # Load connectivity data 
    conn_df = load_data(conn_path)

    # if there is a subject_id column, rename it to match src_subject_id
    if 'subject_id' in conn_df.columns:
        conn_df.rename(columns={'subject_id': 'src_subject_id'}, inplace=True)

    # if subject IDs have underscores, remove them to match src_subject_id format and remove first four characters if they are sub-
    conn_df['src_subject_id'] = conn_df['src_subject_id'].str.replace('_', '', regex=False)
    conn_df['src_subject_id'] = conn_df['src_subject_id'].str.replace('^sub-', '', regex=True)

    # # filter subjects with NaN values in the residualized FC profiles in any conn column
    conn_df = conn_df.dropna(subset=conn_df.columns[1:]) 
    print(f'Connectivity data shape after dropping NaNs: {conn_df.shape}')

    # Merge the connectivity data with the demographic and behavioral data
    # merge on subject_id in FC, and src_subject_id in combined_df
    merged_data = pd.merge(combined_df, conn_df, on='src_subject_id', how='right')
    print(f'Merged data shape: {merged_data.shape}')
    print(merged_data.head())

    return merged_data

In [ ]:
df = merge_abcd_data(
    conn_path=FC_PATH,
    abcd_path=ABCD_DATA_DIR,
    output_path=os.path.join(OUTPUT_DIR, 'abcd_pMTG_FC_data_midb61.csv')
)
print(df.head())

In [ ]:
# drop subjects not recommended for inclusion based on rsfMRI QC
df = df[df['imgincl_rsfmri_include'] == 1]
df.drop(columns=['imgincl_rsfmri_include'], inplace=True)
print(f'Shape after dropping subjects not recommended for inclusion based on rsfMRI QC: {df.shape}')

In [ ]:
# randomly select one subject from each family to avoid relatedness confounds
def select_one_per_family(df):
    df = df.copy()
    df['rel_family_id'] = df['rel_family_id'].astype(str)
    seed = 42
    selected_indices = df.groupby('rel_family_id').apply(lambda x: x.sample(1, random_state=seed).index[0])
    return df.loc[selected_indices].reset_index(drop=True)
df = select_one_per_family(df)
print('Shape after selecting one per family:', df.shape)

In [ ]:
network_labels = {
    'DMN': 1, 'VAN': 7, 'Aud': 12, 'CO': 9, 'PMN': 15,
    'DAN': 5, 'FP': 3, 'PON': 16, 'Sal': 8,
    'SMd': 10, 'SMl': 11, 'Vis': 2, 'Tpole': 13, 'MTL': 14
 } # commented out MIDB labels

# for every column name that contains _fz and matches a network label, add it to a fc_profile list for each subject in src_subject_id
fc_profile_left_columns = []
fc_profile_right_columns = []

for net in network_labels.keys():
    for col in df.columns:
        if f"{net}" in col and "_L_" in col:
            fc_profile_left_columns.append(col)
        if f"{net}" in col and "_R_" in col:
            fc_profile_right_columns.append(col)
print('Left hemisphere FC profile columns:', fc_profile_left_columns)
print('Right hemisphere FC profile columns:', fc_profile_right_columns)
fc_profile_bilateral = fc_profile_left_columns + fc_profile_right_columns
print('Bilateral FC profile columns:', fc_profile_bilateral)

# residualize left and right hemisphere FC profiles based on categorical variable demo_sex_v2 and continuous variable interview_age
def residualize_fc_profiles(df, fc_columns, covariates=['demo_sex_v2', 'interview_age', 'mri_info_softwareversion']):
    df = df.copy()
    
    # Handle missing covariates before encoding (not relevant as there are no missing values)
    for cov in covariates:
        n_missing_before = df[cov].isna().sum()

        if df[cov].dtype == 'object' or cov in ['demo_sex_v2', 'site_id_l', 'mri_info_softwareversion', 'rel_family_id', 'demo_comb_income_v2']:
            # Assume categorical, fill with mode
            mode_val = df[cov].mode()[0]
            df[cov] = df[cov].fillna(mode_val)

            n_filled = n_missing_before - df[cov].isna().sum()
            print(f'Filled {n_filled} missing values in categorical covariate {cov} with mode: {mode_val}')
        else:
            # Assume numeric, fill with mean
            df[cov] = pd.to_numeric(df[cov], errors='coerce')
            mean_val = df[cov].mean()
            df[cov] = df[cov].fillna(mean_val)

            n_filled = n_missing_before - df[cov].isna().sum()
            print(f'Filled {n_filled} missing values in numeric covariate {cov} with mean: {mean_val}')

    # One-hot encode covariates
    df_encoded = pd.get_dummies(df[covariates], drop_first=True)
    
    for col in fc_columns:
        valid_idx = df[col].notnull()
        if valid_idx.sum() == 0:
            df[col + '_resid'] = np.nan
            continue

        X = df_encoded.loc[valid_idx]
        y = df.loc[valid_idx, col]

        model = LinearRegression()
        model.fit(X, y)

        predicted = model.predict(df_encoded)
        df[col + '_resid'] = df[col] - predicted

    return df


In [ ]:
df = residualize_fc_profiles(df, fc_profile_bilateral)

In [ ]:
# remove subjects with values grater than 8 standard deviations from the mean in any of the FC profiles
def remove_outliers(df, columns, threshold=8):
    df = df.copy()
    for col in columns:
        if col in df.columns:
            mean = df[col].mean()
            std = df[col].std()
            upper_limit = mean + threshold * std
            lower_limit = mean - threshold * std
            df = df[(df[col] <= upper_limit) & (df[col] >= lower_limit)]
    return df
df = remove_outliers(df, fc_profile_bilateral)
print(f'Shape after removing outliers: {df.shape}')

In [ ]:
df.to_csv(os.path.join(OUTPUT_DIR, 'abcd_pMTG_FC_data_midb61.csv'), index=False)

Demographics Table Calculations

In [ ]:
race_columns = [col for col in df.columns if 'race' in col]
print(100* df[race_columns].sum()/len(df))
print(df[race_columns].sum())

print(df['ehi1b'].value_counts())
print(df['ehi1b'].value_counts()/len(df))

print(df['demo_sex_v2'].value_counts())
print(df['demo_sex_v2'].value_counts()/len(df))